# Module `dynamic_costs.py`

Ce notebook illustre la logique dynamique locale : les vraies aretes du graphe residuel evoluent autour de leur cout precedent, et certaines peuvent devenir temporairement indisponibles avant un recalcul de la completion metrique.

In [ ]:
import sys
from pathlib import Path
import random

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.dynamic_costs import (
    dynamic_multiplier,
    initialize_dynamic_edge_costs,
    refresh_dynamic_edge_costs,
    sample_dynamic_edge_cost,
)
from cesipath.models import EdgeAttributes, EdgeStatus

In [ ]:
edge = EdgeAttributes(base_cost=10.0, status=EdgeStatus.SURCHARGED, static_surcharge=0.25)
rng = random.Random(42)
previous_cost = None

for step in range(1, 7):
    new_cost = sample_dynamic_edge_cost(
        edge,
        previous_cost=previous_cost,
        rng=rng,
        sigma=0.18,
        mean_reversion_strength=0.35,
        max_multiplier=1.8,
    )
    coefficient = dynamic_multiplier(edge, new_cost)
    print(f"Trajet {step} -> cout={new_cost} ; coefficient={coefficient}")
    previous_cost = new_cost

In [ ]:
edges = {
    (0, 1): EdgeAttributes(base_cost=10.0, status=EdgeStatus.SURCHARGED, static_surcharge=0.25),
    (1, 2): EdgeAttributes(base_cost=8.0),
}
network_rng = random.Random(7)
current_costs = initialize_dynamic_edge_costs(edges)
availability = {key: True for key in current_costs}
print("Etat initial :", current_costs)
current_costs = refresh_dynamic_edge_costs(edges, current_costs, rng=network_rng, sigma=0.18)
print("Apres un deplacement :", current_costs)
print("Disponibilite initiale :", availability)

On voit ici l'effet combine :

- le premier tirage part du cout statique de l'arete ;
- chaque trajet suivant repart du cout dynamique precedent, avec un rappel progressif vers le cout statique ;
- a l'echelle du reseau, toutes les vraies aretes autorisees peuvent etre rafraichies apres un deplacement ;
- l'indisponibilite temporaire d'une arete est geree au niveau du graphe, avec verification de connexite avant coupure ;
- le resultat final ne descend jamais sous le cout de base et ne depasse pas une borne haute configurable.